
# Movie Recommendation System — Evaluator Notebook

This notebook demonstrates a complete movie recommendation system using:

- User-Based Collaborative Filtering
- Item-Based Collaborative Filtering
- Matrix Factorization (SVD)
- Temporal Train-Test Split
- Cold-start handling
- Evaluation using RMSE, MAE, Precision@10, Recall@10
- EDA and visualization


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from src.data_loader import load_ratings
from src.train_test_split import temporal_train_test_split
from src.user_based_cf import train_user_based_model, get_all_predictions as user_predictions
from src.item_based_cf import train_item_based_model, get_all_predictions as item_predictions
from src.matrix_factorization import train_svd_model, get_all_predictions as svd_predictions
from src.evaluator import calculate_rmse, calculate_mae, precision_at_k, recall_at_k
from src.recommender import recommend_top_n
from src.cold_start import recommend_popular_movies

sns.set_style("whitegrid")


## Load Dataset

In [ ]:

data_path = Path("data/ml-100k")
ratings_df = load_ratings(str(data_path))

ratings_df.head()


## Exploratory Data Analysis (EDA)

In [ ]:

print("Number of ratings:", len(ratings_df))
print("Number of users:", ratings_df['user_id'].nunique())
print("Number of movies:", ratings_df['item_id'].nunique())


In [ ]:

plt.figure(figsize=(8,5))
sns.histplot(ratings_df['rating'], bins=5)
plt.title("Rating Distribution")
plt.show()


In [ ]:

user_counts = ratings_df.groupby("user_id")["rating"].count()
plt.figure(figsize=(8,5))
sns.histplot(user_counts, bins=30)
plt.title("Ratings per User")
plt.show()


In [ ]:

movie_counts = ratings_df.groupby("item_id")["rating"].count()
plt.figure(figsize=(8,5))
sns.histplot(movie_counts, bins=30)
plt.title("Ratings per Movie")
plt.show()


## Temporal Train-Test Split

In [ ]:

train_df, test_df = temporal_train_test_split(ratings_df)
print(train_df.shape, test_df.shape)


## Train Models

In [ ]:

user_model = train_user_based_model(train_df)
item_model = train_item_based_model(train_df)
svd_model = train_svd_model(train_df)


## Generate Predictions

In [ ]:

user_preds = user_predictions(user_model, test_df)
item_preds = item_predictions(item_model, test_df)
svd_preds = svd_predictions(svd_model, test_df)


## Evaluation

In [ ]:

def evaluate(name, preds):
    return {
        "Model": name,
        "RMSE": calculate_rmse(preds),
        "MAE": calculate_mae(preds),
        "Precision@10": precision_at_k(preds, k=10),
        "Recall@10": recall_at_k(preds, k=10)
    }

results = pd.DataFrame([
    evaluate("User CF", user_preds),
    evaluate("Item CF", item_preds),
    evaluate("SVD", svd_preds)
])

results


## Model Comparison Visualization

In [ ]:

results.set_index("Model")[["RMSE","MAE"]].plot(kind="bar")
plt.title("Model Comparison")
plt.show()


## Generate Recommendations

In [ ]:

movies_df = ratings_df[['item_id','title']].drop_duplicates()

recommend_top_n(
    model=svd_model,
    train_df=train_df,
    movies_df=movies_df,
    user_id=1,
    n=10
)


## Cold Start Recommendations

In [ ]:

recommend_popular_movies(ratings_df, n=10)
